# 03 — Anomalies & Forecasting\n\nThis notebook detects anomalies (seasonality-aware) and generates a simple ARIMA forecast.\n\n**Proof screenshots to capture:**\n- Anomaly plot with red markers\n- Anomaly table preview\n- Forecast plot with confidence interval\n

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom src.config import DEFAULT_DATA_OUT\nfrom src.analysis.trends import seasonal_decompose_series\nfrom src.analysis.anomaly import detect_anomalies_from_residual\nfrom src.analysis.forecast import forecast_arima\n\ndf = pd.read_csv(DEFAULT_DATA_OUT)\ndf['date'] = pd.to_datetime(df['date'])\nregion = 'Region_A'\nregion_df = df[df['region'] == region].sort_values('date')\n\ntemp_series = region_df.set_index('date')['temp_c'].asfreq('MS')\ndecomp = seasonal_decompose_series(temp_series, period=12)\n\nanoms = detect_anomalies_from_residual(\n    dates=temp_series.index,\n    observed=temp_series.values,\n    residual=decomp['resid'].values,\n    metric_name='temp_c',\n    z_thresh=3.5\n)\nanoms.head()

In [ ]:
plt.figure(figsize=(12,4))\nplt.plot(temp_series.index, temp_series.values, label='Observed')\nplt.scatter(pd.to_datetime(anoms['date']), anoms['observed'], color='red', s=35, label='Anomaly')\nplt.title(f'Temperature anomalies — {region}')\nplt.xlabel('Date'); plt.ylabel('Temperature (°C)'); plt.grid(True, alpha=0.3)\nplt.legend()\nplt.show()

In [ ]:
forecast_df = forecast_arima(temp_series.dropna(), steps=24)\nforecast_df.head()

In [ ]:
plt.figure(figsize=(12,4))\nplt.plot(temp_series.index, temp_series.values, label='History')\nfc_dates = pd.to_datetime(forecast_df['date'])\nplt.plot(fc_dates, forecast_df['forecast'], label='Forecast', linewidth=2)\nplt.fill_between(fc_dates, forecast_df['lower'], forecast_df['upper'], alpha=0.2, label='95% CI')\nplt.title(f'Temperature forecast — {region}')\nplt.xlabel('Date'); plt.ylabel('Temperature (°C)'); plt.grid(True, alpha=0.3)\nplt.legend()\nplt.show()